# **CA4** - Telecom Customer Churn Classification

# **Group 6** - Narges Zareigolbaghi & Kachina Aschenberg Øwre 
#**Kaggle names - nargeszareigolbaghi & kachasch**

RULES AND CONTEXT OF THE COMPETITION

1. The general workflow is similar to that of CA3. As opposed to CA3, however, you are now expected to use cross-validation instead of multiple train/validation splits. You can use the scikit-learn implementations for cross-validation.

2. You have access to a range of classifiers, kernels, and L1/L2-regularisation covered in DAT200. These can be combined freely. You can also use the ensemble models presented in the lectures. Use the corresponding scikit-learn implementations.

3. Classifier selection and hyperparameter tuning are the core of this exercise. You are expected to make use of all of the following components:

   - Hyperparameter tuning (using scikit-learn tools from Chapter 6)
   - Scikit-learn Pipelines
   - Cross-validation
   - Some form of dimension reduction (e.g. PCA or SFS)
   - A confusion matrix for your final classifier, using a 60/40 train-test split on the entire training dataset
   - Macro-averaged F1-score as the primary evaluation metric (other metrics are encouraged too)

   You must build at least two pipelines combining preprocessing with a classifier. One pipeline must use a kernel, one must use regularisation, and at least one must include a dimension reduction step.

4. You may try other models. If your best-performing Kaggle submission is not one of the two mandatory pipelines, also provide the code for training, tuning, and prediction for that classifier.

5. You may engineer new features if you wish.

6. If you are unsure how to structure your notebook, follow these steps:
   Imports → Read the data → Data exploration & visualisation → Data cleaning → Data exploration & visualisation after cleaning → Preprocessing → Evaluating models/hyperparameters (cross-validation) → Kaggle submission.

DATA DESCRIPTION

The dataset contains information about the customers of a mobile phone operator. Each row is one customer. The features are:

   - CustomerId – unique ID of the customer (not a feature for modelling)
   - State – US state where the customer lives (two-letter code)
   - AccountLength – how long the customer has had an account
   - AreaCode – telephone area code of the customer
   - InternationalPlan – whether the customer has an international calling plan (yes/no)
   - VoiceMailPlan – whether the customer has a voice mail plan (yes/no)
   - NumVoiceMailMessages – number of voice mail messages
   - DayMinutes, DayCalls, DayCharge – total minutes, number of calls and total charge for daytime calls
   - EveMinutes, EveCalls, EveCharge – the same for evening calls
   - NightMinutes, NightCalls, NightCharge – the same for night calls
   - IntlMinutes, IntlCalls, IntlCharge – the same for international calls
   - CustomerServiceCalls – number of calls the customer has made to customer service

Target variable: Churn
   - 0 = customer stays
   - 1 = customer leaves the company

BACKGROUND

You have been hired by a mobile phone operator in the United States. The company is experiencing customer churn, where customers cancel their subscription and move to a competing operator. Retaining existing customers is often significantly cheaper than acquiring new ones.

Your task is to develop a machine learning model capable of predicting whether a customer will leave the company based on information about their subscription plan and phone usage. Accurate churn prediction models allow companies to identify customers who may leave and take proactive measures to retain them.

### 1. Imports

In [2]:
import numpy as np                 # numbers and arrays
import pandas as pd                # tables (DataFrames)
import matplotlib.pyplot as plt    # basic plotting
import seaborn as sns              # nicer statistical plots on top of matplotlib
import sklearn as skl              # machine learning library

from sklearn.model_selection import train_test_split # splits data into training and test sets
from sklearn.preprocessing import StandardScaler # rescales features to have mean 0 and variance 1
from sklearn.linear_model import Perceptron, LogisticRegression # linear classifiers, baseline for comparison and neural networks
from sklearn.svm import SVC     # support vector machine, a more powerful linear classifier
from sklearn.tree import DecisionTreeClassifier # decision tree, a non-linear classifier
from sklearn.ensemble import RandomForestClassifier     # random forest, a more powerful non-linear classifier
from sklearn.neighbors import KNeighborsClassifier  # k-nearest neighbors, a non-linear classifier

### 2. Reading data

In [ ]:
train = pd.read_csv("trainCA4DAT200.csv")                 # 19 features + CustomerId + Churn lables
test = pd.read_csv("testCA4DAT200.csv")                   # 19 features + + CustomerId - no Churn lables 
sample = pd.read_csv("sample_submissionCA4DAT200.csv")    # the format Kaggle expects
print("train:", train.shape, " test:", test.shape, " sample:", sample.shape)
# training data has 3500 rows x 21 columns

# The feature columns: everything except the target (Churn) and CustomerId
print(train.head())
features = [col for col in train.columns if col not in ("Churn", "CustomerId")]
print(len(features), "features:", features)

train: (3500, 21)  test: (1500, 20)  sample: (1500, 2)
   CustomerId State  AccountLength  AreaCode InternationalPlan VoiceMailPlan  \
0      952920    NJ             87       510                no            no   
1      895150    RI            114       510                no           yes   
2      636987    ND             55       415                no            no   
3      538188    DC            100       510               yes            no   
4      947531    AL             74       408                no           yes   

   NumVoiceMailMessages  DayMinutes  DayCalls  DayCharge  ...  EveCalls  \
0                     0       223.2       109      37.94  ...        86   
1                    28       225.8        94      38.39  ...       117   
2                     0       180.1        77      30.62  ...        98   
3                     0        68.5       110      11.65  ...       115   
4                    46       277.7       109      47.21  ...       114   

   EveCharge 

### 3. Data exploration and visualisation

In [ ]:
# Explore the training set
train.info()                       # shows entries, column types and memory usage
display(train.describe().T)              # .T turns the table: one row per feature
print(train["Churn"].value_counts())  # Count of customers who stay (0) or leave (1)
print("Missing values:", train.isnull().sum().sum()) # total number of missing values in the training set
print("Duplicate rows (ignoring Id):", train.duplicated(subset=features + ["Churn"]).sum()) # count of duplicate rows in the training set, ignoring the Id column

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   CustomerId            3500 non-null   int64  
 1   State                 3500 non-null   str    
 2   AccountLength         3500 non-null   int64  
 3   AreaCode              3500 non-null   int64  
 4   InternationalPlan     3500 non-null   str    
 5   VoiceMailPlan         3500 non-null   str    
 6   NumVoiceMailMessages  3500 non-null   int64  
 7   DayMinutes            3500 non-null   float64
 8   DayCalls              3500 non-null   int64  
 9   DayCharge             3500 non-null   float64
 10  EveMinutes            3500 non-null   float64
 11  EveCalls              3500 non-null   int64  
 12  EveCharge             3500 non-null   float64
 13  NightMinutes          3500 non-null   float64
 14  NightCalls            3500 non-null   int64  
 15  NightCharge           3500 non-n

KeyError: 'quality'

In [ ]:
# distribution of wine quality  
sns.countplot(data=train, x="quality")
plt.xlabel("wine quality (0 = Bad, 1 = Good)")
plt.ylabel("no. of wines")
plt.title("distribution of wine quality")
plt.show()
# Wine quality between good and bad is eqally distributed 

- Alcohol is the dominant single predictor at +0.452, so higher alcohol associates with good wine.
- Volatile acidity is the strongest negative at −0.302. It's the compound that makes wine taste of vinegar, so this assosiates with bad wine.
- No feature is strong enough to classify on its own. The largest is 0.452; a correlation that size explains roughly 20% of the variation, so this needs a model combining several features.
- All features and values ar valuable, because Pearson only detects linear relationships and a weak score here doesn't mean a feature is useless in combination.

### 4. Data cleaning

In [ ]:
data = train.drop(columns=["Id"]) # drop the Id column, if it was still there would be no duplicates, because every row has a unique Id.

# Check for duplicate rows with conflicting labels
duplicates = data.duplicated(subset=features, keep=False) # mark all rows that have the same features as another row
conflicts = data[duplicates].groupby(features)["quality"].nunique() # count how many different quality values each group of duplicate features has
print("Duplicate groups with conflicting labels:", (conflicts > 1).sum())

# Keep the first copy of every exact duplicate
data = data.drop_duplicates()
print("Rows after removing exact duplicates:", len(data))

In [ ]:
print("Negative values:", (data[features] < 0).sum().sum()) # count of negative values in the features
print("pH outside 2.5 to 4.5:", ((data["pH"] < 2.5) | (data["pH"] > 4.5)).sum()) # count of pH values outside the range 2.5 to 4.5
print("Alcohol outside 5 to 16 %:", ((data["alcohol"] < 5) | (data["alcohol"] > 16)).sum()) # count of alcohol values outside the range 5 to 16 %

In [ ]:
z = (data[features] - data[features].mean()) / data[features].std() # compute the z-score for each feature in each row
print((z.abs() > 3).sum())                         # outliers per feature
outlier_rows = (z.abs() > 3).any(axis=1)           # True if ANY feature has |z| > 3
print("Rows with |z| > 3 in any feature:", outlier_rows.sum()) # count of rows with outliers in any feature

### 5. Data exploration and visualisation after cleaning

In [ ]:
# How much did cleaning change the data?
print("Rows before:", len(train), " after:", len(data))
print(data["quality"].value_counts(normalize=True).round(3))

# The same feature before and after, drawn on one pair of axes
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flatten(), features):
    sns.kdeplot(train[col], ax=ax, label="before", fill=True, alpha=.3)
    sns.kdeplot(data[col],  ax=ax, label="after",  fill=True, alpha=.3)
    ax.set_title(col)
    ax.set_xlabel("")
axes.flatten()[-1].set_visible(False)   # hide the empty 12th panel
axes.flatten()[0].legend()
fig.suptitle("Feature distributions before and after cleaning")
plt.tight_layout()
plt.show()

# Did any correlation with quality move?
before = train[features + ["quality"]].corr()["quality"].drop("quality")
after  = data[features + ["quality"]].corr()["quality"].drop("quality")
comparison = pd.DataFrame({"before": before, "after": after})
comparison["change"] = comparison["after"] - comparison["before"]
print(comparison.round(3).sort_values("after"))


Distribution: the balance was 50.1/49.9 before and 50.1/49.9 after. The duplicates that came out were split evenly between the classes.

### 6. Data preprocessing

In [ ]:
# Makning the feature matrix and target vector for training the model

X = data[features].to_numpy() # one row per wine, one column per feature (760, 11)
y = data["quality"].to_numpy() # one row per wine, one column per target variable (760,)

In [ ]:
scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=features) # standardize the features
plt.figure(figsize=(14, 5))
sns.boxplot(data=scaled)
plt.xticks(rotation=45, ha="right")
plt.title("All features after standardisation")
plt.ylabel("Standard deviations from the mean")
plt.show()

In [ ]:
def evaluate(make_model, X, y, n_splits=50, scale=True, drop_outliers=False):
    """Score a model by averaging over many random train/validation splits.

    The course rules forbid GridSearchCV and cross_val_score, so this is our own replacement. 
    We repeat the split n_splits times with a different random_state each pass and average the results. 
    Lecture 12 introduces this as the stand-in for k-fold cross-validation.

    Because random_state counts 0, 1, 2, ..., every model passed to this
    function sees the identical set of splits. That is what makes comparing
    different classifiers in section 7 a fair test.

    Parameters
    ----------
    make_model : callable
        A no-argument function that RETURNS a new, unfitted model, for example
        lambda: LogisticRegression(C=1, max_iter=1000). It has to be a function
        rather than a model object so that every split gets a fresh estimator
        with no weights left over from the previous pass.
    X : ndarray, shape (n_samples, n_features)
        The cleaned feature matrix.
    y : ndarray, shape (n_samples,)
        The 0/1 quality labels.
    n_splits : int, default 50
        How many random splits to average over. Drop to about 10 while
        exploring a large grid, then raise it back for the numbers reported here.
    scale : bool, default True
        Standardise the features. True for distance- and gradient-based models
        (KNN, SVC, logistic regression, perceptron). False for decision trees
        and random forests, which split one feature at a time on a threshold
        and are therefore unaffected by rescaling.
    drop_outliers : bool, default False
        Remove training rows where any feature lies more than 3 standard
        deviations from its mean. Whether this helps is an empirical question,
        so it is an option to measure rather than something applied by default.

    Returns
    -------
    (float, float, float)
        Mean training accuracy, mean validation accuracy, and the standard
        deviation of validation accuracy across the splits. The gap between the
        first two shows overfitting; the third shows how stable the model is.
    """
    train_scores, val_scores = [], []
    for randome_split in range(n_splits):
        # 1. A new random split, the same one for every model
        # A different split each pass, but a reproducible one, since randome_split is the seed. 
        # stratify=y keeps the 50/50 class balance in both halves
        X_tr, X_va, y_tr, y_va = train_test_split(
            X, y, test_size=0.3, random_state=randome_split, stratify=y)
    
        # 2. Remove outliers from the training part only makes the test easier and flattens the score.
        if drop_outliers:
            z_tr = (X_tr - X_tr.mean(axis=0)) / X_tr.std(axis=0)
            keep = (np.abs(z_tr) < 3).all(axis=1)
            X_tr, y_tr = X_tr[keep], y_tr[keep]

        # 3. Scale with a scaler fitted on the training part only
        # fit_transform learns the mean and standard deviation; transform only applies them. Fitting on
        # the full data instead would let validation rows influence the transformation used on training rows. 
        # That is data leakage: it runs without any error and quietly makes the validation score optimistic.
        if scale:
            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr)
            X_va = sc.transform(X_va)
        

        # 4. A fresh model, trained and scored
        # A brand new model every pass. make_model() builds it here
        # so no weights survive from the previous split.
        model = make_model()
        model.fit(X_tr, y_tr)
        train_scores.append(model.score(X_tr, y_tr))
        val_scores.append(model.score(X_va, y_va))

    # np.std measures spread across the splits: a large value means the model's
    # accuracy depends heavily on which rows it happened to train on.
    return np.mean(train_scores), np.mean(val_scores), np.std(val_scores)

tr, va, sd = evaluate(lambda: LogisticRegression(max_iter=1000), X, y)
print(f"train {tr:.3f}, validation {va:.3f} ± {sd:.3f}")

### 7. Evaluating different models and hyperparameters (using train/test splits)

In [ ]:
results = []   # run ONCE; every tuning cell appends to this

In [ ]:

# Tune the learning rate (eta0) for the Perceptron
for eta in [0.0001, 0.001, 0.01, 0.1, 1.0]:
    tr, va, sd = evaluate(lambda: Perceptron(eta0=eta, max_iter=100, random_state=1), X, y)
    results.append({"model": "Perceptron", "settings": f"eta0={eta}",
                    "train": tr, "val": va, "std": sd})

In [ ]:
# Tune the regularization strength (C) for the Logistic Regression
for c in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    tr, va, sd = evaluate(lambda: LogisticRegression(C=c, max_iter=1000), X, y)
    results.append({"model": "Logistic regression", "settings": f"C={c}",
                    "train": tr, "val": va, "std": sd})

In [ ]:
# Tune the number of neighbors (k) and distance metric (p) for the KNN
for k in range(1, 52, 2):          # odd k from 1 to 51
    for p in [1, 2]:               # 1 = Manhattan, 2 = Euclidean
        tr, va, sd = evaluate(lambda: KNeighborsClassifier(n_neighbors=k, p=p), X, y)
        results.append({"model": "KNN", "settings": f"k={k}, p={p}", "k": k, "p": p,
                        "train": tr, "val": va, "std": sd})

# Training vs validation accuracy as k grows (Euclidean distance)
res = pd.DataFrame(results)
knn_p2 = res[(res["model"] == "KNN") & (res["p"] == 2)]
plt.plot(knn_p2["k"], knn_p2["train"], label="Train")
plt.plot(knn_p2["k"], knn_p2["val"], label="Validation")
plt.xlabel("k")
plt.ylabel("Mean accuracy over 50 splits")
plt.title("KNN: training and validation accuracy as k grows")
plt.legend()
plt.show()

In [ ]:
# DecisionTree
from sklearn.tree import DecisionTreeClassifier

# evaluating different maximum tree depths using the repeated train/validation splits  in evaluate()
for depth in [2, 3, 4, 5, 6, 8, 10, None]:

    tr, va, sd = evaluate(
        lambda: DecisionTreeClassifier(
            criterion='gini',
            max_depth=depth,
            random_state=1
        ),
        X, y,
        scale=False
    )
# saving the scores 
    results.append({
        "model": "Decision Tree",
        "settings": f"max_depth={depth}",
        "train": tr,
        "val": va,
        "std": sd
    })

In [ ]:
# Random forest 

from sklearn.ensemble import RandomForestClassifier

# looping through combinations of n_estimators and max_depth and evaluating each Random Forest
for depth in [3, 5, 10, None]:
    for n in [50, 100, 200]:

        tr, va, sd = evaluate(
            lambda: RandomForestClassifier(
                n_estimators=n,
                criterion='gini',
                max_depth=depth,
                random_state=1,
                n_jobs=-1
            ),
            X, y,
            scale=False
        )
# saving the scores 
        results.append({
            "model": "Random Forest",
            "settings": f"n_estimators={n}, max_depth={depth}",
            "train": tr,
            "val": va,
            "std": sd
        })

In [ ]:
# SVC
from sklearn.svm import SVC

# Test linear and rbf kernels, each with 5 different regularization (c) values
for kernel in ['linear', 'rbf']:
    for c in [0.01, 0.1, 1, 10, 100]:

        tr, va, sd = evaluate(
            lambda: SVC(
                kernel=kernel,
                C=c,
                gamma='scale',
                random_state=1
            ),
            X, y
        )
# saving the scores 
        results.append({
            "model": "SVM",
            "settings": f"kernel={kernel}, C={c}",
            "train": tr,
            "val": va,
            "std": sd
        })

In [ ]:
# Comparing the 6 classifiers 

results_df = pd.DataFrame(results)

best_models = (
    results_df
    .sort_values("val", ascending=False)
    .drop_duplicates(subset="model")
    .sort_values("val", ascending=False)
)

print(best_models.to_string(index=False))

# Random forest is currently the best classifier with 0.755614 accurasy, but it also has a big gap between val and train (about 10%) which can suggest some overfitting. Still it can be a good candidate model. 
# Logistic regression has the least val and train gap (about 1%) which suggests if generalizes better although having a slightly lower accuracy of 0.743158. It can also be a potential candidate. 
# Perceptron has the lowest validation accuracy (0.663158) and the biggest standard deviasion (0.054060)

### 8. Final evaluation

The tuning in section 7 trained on 70% of the data in each split and held 30% back to measure with. That measuring is now finished, so the final model is trained on all 760 cleaned rows: more data gives a better model, and there is nothing left to hold out for.

The random forest needs no scaling. A tree splits one feature at a time on a threshold, so rescaling a feature cannot change which split is best. That removes the scaler from the final pipeline entirely, and with it the most common way of leaking information into a test set.

In [ ]:
best_model = RandomForestClassifier(
    n_estimators=200, max_depth=5, random_state=1, n_jobs=-1
)

X_full = data[features].to_numpy()        # all 760 cleaned rows
X_test = test[features].to_numpy()        # same 11 columns, same order, untouched

best_model.fit(X_full, y)

print("trained on", X_full.shape[0], "rows x", X_full.shape[1], "features")
print("accuracy on the full training set:", round(best_model.score(X_full, y), 4))
print("test matrix ready:", X_test.shape)

importance = (pd.Series(best_model.feature_importances_, index=features)
                .sort_values(ascending=False).round(3))
print("\nwhich features the forest actually used:")
print(importance.to_string())

# Train our best classifier on the full cleaned training dataset
final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=1
)

final_model.fit(X, y)

# Predict the Kaggle test data
X_test = test[features].to_numpy()
predictions = final_model.predict(X_test)

# Use Kaggle's sample submission to preserve the required format
submission = sample.copy()
submission.iloc[:, -1] = predictions

# Save the submission file
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)

### 9. Kaggle submission

The submission format is taken from the example code on the competition's Overview page: a column named `Quality`, with the DataFrame index named `ID`, written out so that the row numbers 0 to 208 become the ID column.

This is worth stating because two other sources in the folder point elsewhere. `sample_submission.csv` carries the right IDs but spells the target `quality` in lower case, and an upload in that form is rejected. The `Id` column in `test.csv`, which runs 1117, 1411 and so on, is not what the scorer keys on either.

No cleaning is applied to `test.csv`. Every row has to receive a prediction, so removing duplicates or outliers there would leave gaps the submission cannot have. Selecting `test[features]` preserves the original row order, so prediction *i* belongs to row *i*.

In [ ]:
# Submission format copied from the example code on the competition's Overview page:
#     y_test = pd.DataFrame(y_test, columns=["Quality"])
#     y_test.index.name = "Id"
#     y_test[['Quality']].to_csv("submission.csv")
y_pred_kaggle = best_model.predict(X_test)

y_test = pd.DataFrame(y_pred_kaggle, columns=["quality"])
y_test.index.name = "Id"                     # row numbers 0..208 become the ID column
y_test[["quality"]].to_csv("submission.csv")

# checks before uploading
check = pd.read_csv("submission.csv")
sample = pd.read_csv("sample_submission.csv")
print("header:            ", ",".join(check.columns))
print("shape:             ", check.shape, "  (expected (209, 2))")
print("IDs match sample:  ", check["Id"].tolist() == sample["ID"].tolist())
print("missing values:    ", int(check.isna().sum().sum()))
print("predicted classes: ", check["quality"].value_counts().to_dict())
print()
print(check.head().to_string(index=False))